# Atraso de presentación

Los zips van de 2018q1 a 2024q4 y los notebooks filtran por `period`, no por zip. Este notebook justifica ese rango:

- **Antes de 2018 no hace falta nada.** Un documento se presenta siempre después de su `period`. Ningún zip de 2017 trae un documento con `period` de 2018 en adelante.
- **Después de 2023, ¿alcanza con un año (2024)?** Depende del atraso entre el cierre del documento (`period`) y su presentación (`filed`). Este notebook lo mide.

Solo usa documentos con `period` de 2018 a 2022. Los zips cubren por lo menos 2 años después de esas fechas, así que se ven atrasos de hasta 730 días. Los de 2023 se excluyen porque los zips cubren menos tiempo después.

In [1]:
import pandas as pd

sub = pd.read_csv("data/sub.csv", dtype=str)

period = pd.to_datetime(sub["period"], format="%Y%m%d")
filed = pd.to_datetime(sub["filed"], format="%Y%m%d")
sub["delay"] = (filed - period).dt.days
sub = sub[period.dt.year.between(2018, 2022)]

sub["delay"].describe()

count    5676.000000
mean       55.695560
std        64.808843
min        14.000000
25%        36.000000
50%        43.000000
75%        54.000000
max      1520.000000
Name: delay, dtype: float64

## Documentos con atraso mayor a cada umbral

In [2]:
pd.DataFrame(
    {
        "documents": [(sub["delay"] > d).sum() for d in [90, 180, 365, 730]],
        "companies": [sub.loc[sub["delay"] > d, "cik"].nunique() for d in [90, 180, 365, 730]],
    },
    index=pd.Index([90, 180, 365, 730], name="more than (days)"),
)

,documents,companies
more than (days),,
90,393,143
180,117,36
365,48,12
730,8,1


## Documentos con más de un año de atraso

In [3]:
sub.loc[sub["delay"] > 365, ["name", "sector", "form", "period", "filed", "delay"]].sort_values("delay")

,name,sector,form,period,filed,delay
4024,BARREL ENERGY INC.,energy,10-Q,20200331,20210408,373
4592,SUSTAINABLE PROJECTS GROUP INC.,energy,10-Q,20200930,20211012,377
2109,SUSTAINABLE PROJECTS GROUP INC.,energy,10-Q,20180831,20190913,378
4034,GOLDEN ROYAL DEVELOPMENT INC.,energy,10-Q,20200331,20210413,378
5403,PETROLIA ENERGY CORP,energy,10-Q,20210630,20220713,378
5108,"BRISSET BEER INTERNATIONAL, INC.",energy,10-K,20210531,20220621,386
2635,"VICTORY OILFIELD TECH, INC.",energy,10-K,20181231,20200127,392
4264,"VICTORY OILFIELD TECH, INC.",energy,10-Q,20200630,20210810,406
3715,"VICTORY OILFIELD TECH, INC.",energy,10-K,20191231,20210209,406
3454,"VICTORY OILFIELD TECH, INC.",energy,10-Q,20190930,20201116,413


## ¿Esas empresas tienen los 24 trimestres?

Si no los tienen, quedan afuera del panel igual, y un zip más (2025) no cambia la cantidad de sobrevivientes. `quarters` cuenta los trimestres con presentación en 2018–2023, con el mismo criterio que `survivorship_bias.ipynb`.

In [4]:
all_docs = pd.read_csv("data/sub.csv", dtype=str)
all_docs["quarter"] = pd.to_datetime(all_docs["period"], format="%Y%m%d").dt.to_period("Q")
all_docs = all_docs[all_docs["quarter"].between(pd.Period("2018Q1"), pd.Period("2023Q4"))]

late = sub.loc[sub["delay"] > 365, "cik"].unique()
all_docs[all_docs["cik"].isin(late)].groupby("cik").agg(name=("name", "last"), quarters=("quarter", "nunique"))

,name,quarters
cik,,
1295961,QUANTUM ENERGY INC.,15
1368637,PETROLIA ENERGY CORP,23
1389518,"TONGJI HEALTHCARE GROUP, INC.",10
1448695,SYLIOS CORP,2
1493212,PHOENIX LIFE SCIENCES INTERNATIONAL LIMITED.,4
1495648,POWER AMERICAS RESOURCE GROUP LTD.,19
1500305,SUSTAINABLE PROJECTS GROUP INC.,24
1505497,"BIORESTORATIVE THERAPIES, INC.",24
1631463,BARREL ENERGY INC.,10


## Resultado

- La mediana del atraso es de 43 días. 48 de 5.676 documentos (0,8%, 12 empresas) se presentan con más de un año de atraso, y 8 (1 empresa) con más de dos años.
- De esas 12 empresas, 9 no tienen los 24 trimestres. Las 3 que sí los tienen (BioRestorative Therapies, Sustainable Projects Group y Victory Clean Energy) se presentaron tarde, pero sus documentos están en los zips.
- Conclusión: los zips de 2018q1 a 2024q4 alcanzan. Un zip de 2017 no aporta documentos, y un zip de 2025 solo agregaría documentos de 2023 con más de un año de atraso, que según 2018–2022 son pocos.